# NA Inversion Example (updated physics)

This notebook demonstrates kinematic inversion using the Neighbourhood Algorithm (NA) with the updated physical implementation:

$$M_0 = \mu(z) \times A \times slip$$

**Workflow:**
1. Load configuration and forward model
2. Generate controlled synthetic observed waveforms from a 7-parameter ellipse model
3. Run NA search
4. Analyze best model and compare $M_0$/$M_w$
5. Visualize convergence and export results

**Parameters inverted:**
- a1, a2: Ellipse semi-axes (km)
- theta: Rotation angle (x π)
- np, tp: Center position
- dmax: Maximum slip (m)
- vr: Rupture velocity (km/s)

## Step 1: Load configuration

## Step 2: Load REAL DATA and check it.

This apply for root / 'DATA' / 'real_disp_x'

Plot the data to visualize

In [6]:
# Initialize forward model and use midpoint model parameters
fwd = AxitraForwardModel(str(INPUT_CTL))

# Use midpoint of parameter ranges from input.ctl
midpoint_model = np.array([
    0.5 * (float(p.min_val) + float(p.max_val))
    for p in cfg.inversion_params.parameters
], dtype=float)

print(f"✓ Forward model initialized")
print(f"  AXITRA dir: {fwd.axitra_dir}")
print(f"\n✓ Using midpoint model parameters:")
param_names = ['a1', 'a2', 'theta', 'np', 'tp', 'dmax', 'vr']
for name, val in zip(param_names, midpoint_model):
    print(f"  {name:6s} = {val:.4f}")

# Build geometry
geometry = fwd.build_geometry_with_ellipse_slip(midpoint_model)
m0, mw = fwd.estimate_total_moment_and_mw(midpoint_model, geometry)
print(f"\n✓ Geometry built:")
print(f"  Total moment M0 = {m0:.3e} N.m")
print(f"  Moment magnitude Mw = {mw:.2f}")


✓ Forward model initialized
  AXITRA dir: /home/alex/elliptical-rupture-updated/Kinematic_inversion/AXITRA2024

✓ Using midpoint model parameters:
  a1     = 7.5000
  a2     = 7.5000
  theta  = 1.0000
  np     = 0.5000
  tp     = 0.5000
  dmax   = 2.0000
  vr     = 2.0000

✓ Geometry built:
  Total moment M0 = 1.838e+19 N.m
  Moment magnitude Mw = 6.78


In [ ]:
# Create output directory
from datetime import datetime

output_dir = root / 'output' / f'na_inversion_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Output directory created: {output_dir}")

# Export results using the built-in method
result.export_results(output_dir / 'inversion_results.json')
print(f"✓ Results exported to: {output_dir / 'inversion_results.json'}")

# Save additional analysis
import json

summary = {
    'timestamp': datetime.now().isoformat(),
    'total_models': len(result.all_models),
    'iterations': int(na_config.n_iterations),
    'best_misfit': float(result.best_model.misfit),
    'best_iteration': int(result.best_model.iteration),
    'best_model': result.best_model.model.tolist(),
    'best_model_m0': float(m0_best),
    'best_model_mw': float(mw_best),
    'initial_model': midpoint_model.tolist(),
    'initial_misfit': float(misfit_test),
    'search_time_seconds': float(t_elapsed),
    'misfit_improvement_percent': float(improvement),
    'config': {
        'freq1': float(freq1),
        'freq2': float(freq2),
        't0': float(cfg.ellipse.t0),
        'time_window_s': 20.0,
    }
}

summary_path = output_dir / 'inversion_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✓ Summary saved to: {summary_path}")

# Save waveform comparison
waveform_data = {
    'time': time.tolist(),
    'observed': observed.tolist(),
    'synthetic_best': synthetic_best_filtered.tolist(),
    'residual': (observed - synthetic_best_filtered).tolist(),
}

waveform_path = output_dir / 'waveform_comparison.json'
with open(waveform_path, 'w') as f:
    json.dump(waveform_data, f)

print(f"✓ Waveform data saved to: {waveform_path}")

# Print summary
print("\n" + "=" * 70)
print("INVERSION SUMMARY")
print("=" * 70)
print(f"Total models evaluated: {len(result.all_models)}")
print(f"Search iterations: {na_config.n_iterations}")
print(f"Search time: {t_elapsed:.1f} seconds")
print(f"\nInitial model misfit: {misfit_test:.6f}")
print(f"Best model misfit:    {result.best_model.misfit:.6f}")
print(f"Improvement:          {improvement:.1f}%")
print(f"\nBest model:")
for name, val in zip(param_names, result.best_model.model):
    print(f"  {name:6s} = {val:.4f}")
print(f"\nBest model moment:")
print(f"  M0 = {m0_best:.3e} N.m")
print(f"  Mw = {mw_best:.2f}")
print(f"\nResults saved to: {output_dir}")
print("=" * 70)


In [ ]:
# Generate and visualize synthetic waveforms from best model
print("Generating synthetic waveforms from best model...")

# Build geometry with best model
geometry_best = fwd.build_geometry_with_ellipse_slip(result.best_model.model)

# Compute Green functions and convolve
axitra_best = fwd.build_axitra(geometry_best, latlon=False, freesurface=True)
ap_best = fwd.green(axitra_best, quiet=True)

result_best = fwd.conv(ap_best, geometry_best, source_type=1, t0=float(cfg.ellipse.t0), quiet=True)

if isinstance(result_best, tuple):
    _, sx_best, sy_best, sz_best = result_best
    synthetic_best = np.array([sx_best, sy_best, sz_best], dtype=float)
    synthetic_best = np.transpose(synthetic_best, (1, 0, 2))  # (nsta, 3, npts)
else:
    synthetic_best = result_best

# Apply filtering
synthetic_best_filtered = bandpass_filter_waveforms(
    synthetic_best, time,
    freq1=freq1, freq2=freq2,
    corners=4, zerophase=True
)

print("✓ Synthetic waveforms from best model generated")

# Calculate misfit for best model
misfit_best = misfit_calc.l2_misfit(synthetic_best_filtered)
print(f"  Misfit (best model): {misfit_best:.6f}")

# Visualize comparison
station_idx = 0
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

components = ['X', 'Y', 'Z']

# Top row: Time series
for icomp in range(3):
    ax = axes[0, icomp]
    ax.plot(time, observed[station_idx, icomp], 'g-', linewidth=2, label='Observed', alpha=0.8)
    ax.plot(time, synthetic_best_filtered[station_idx, icomp], 'r--', linewidth=1.5, label='Best synthetic', alpha=0.8)
    ax.set_title(f'Station {station_idx+1} - {components[icomp]} Component')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Bottom row: Residuals
for icomp in range(3):
    ax = axes[1, icomp]
    residual = observed[station_idx, icomp] - synthetic_best_filtered[station_idx, icomp]
    ax.plot(time, residual, 'orange', linewidth=1.5)
    ax.axhline(0, color='k', linestyle=':', alpha=0.5)
    ax.set_title(f'Residual - {components[icomp]}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Best Model Fit (Misfit: {misfit_best:.6f})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Best model visualization complete")


In [ ]:
# Extract all models and misfits for analysis
all_models = np.array([m.model for m in result.all_models])
all_misfits = np.array([m.misfit for m in result.all_models])
all_iterations = np.array([m.iteration for m in result.all_models])

print(f"✓ Extracted {len(all_models)} models from search")
print(f"  Misfit range: [{all_misfits.min():.6f}, {all_misfits.max():.6f}]")
print(f"  Iterations: {all_iterations.min()} to {all_iterations.max()}")

# Plot convergence
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Misfit convergence (all models)
axes[0, 0].scatter(range(len(all_misfits)), all_misfits, alpha=0.6, s=30)
axes[0, 0].axhline(result.best_model.misfit, color='r', linestyle='--', linewidth=2, label='Best')
axes[0, 0].set_xlabel('Model Index')
axes[0, 0].set_ylabel('Misfit (L2)')
axes[0, 0].set_title('Convergence: All Models')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Misfit vs iteration
axes[0, 1].scatter(all_iterations, all_misfits, alpha=0.6, s=30, c=all_iterations, cmap='viridis')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('Misfit (L2)')
axes[0, 1].set_title('Misfit by Iteration')
axes[0, 1].grid(True, alpha=0.3)

# Best misfit evolution by iteration
best_per_iteration = []
for it in range(int(all_iterations.max()) + 1):
    mask = all_iterations == it
    if mask.any():
        best_per_iteration.append(np.min(all_misfits[mask]))

axes[1, 0].plot(range(len(best_per_iteration)), best_per_iteration, 'o-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('Best Misfit in Iteration')
axes[1, 0].set_title('Best Misfit Evolution')
axes[1, 0].grid(True, alpha=0.3)

# Parameter space: best models
n_best = min(20, len(result.all_models))
best_indices = np.argsort(all_misfits)[:n_best]
best_models_subset = all_models[best_indices]
best_misfits_subset = all_misfits[best_indices]

axes[1, 1].scatter(best_models_subset[:, 0], best_models_subset[:, 1], 
                   c=best_misfits_subset, cmap='RdYlGn_r', s=50, alpha=0.8)
axes[1, 1].scatter(result.best_model.model[0], result.best_model.model[1], 
                   color='red', s=200, marker='*', label='Best model', edgecolor='black', linewidth=2)
axes[1, 1].set_xlabel('a1 (km)')
axes[1, 1].set_ylabel('a2 (km)')
axes[1, 1].set_title('Parameter Space (a1 vs a2)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('NA Inversion Convergence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Convergence analysis plot complete")


In [ ]:
# Run NA search
print("=" * 70)
print("STARTING NEIGHBOURHOOD ALGORITHM SEARCH")
print("=" * 70)

import time as time_module

t0 = time_module.time()

result = na_inversion.run_na_search(na_config=na_config)

t_elapsed = time_module.time() - t0

print("=" * 70)
print("NA SEARCH COMPLETE")
print("=" * 70)
print(f"\nSearch completed in {t_elapsed:.1f} seconds")
print(f"\n✓ Best model found:")
print(f"  Misfit: {result.best_model.misfit:.6f}")
print(f"  Iteration: {result.best_model.iteration}")

# Show best model parameters
print(f"\n✓ Best model parameters:")
best_params = result.best_model.model
for name, val in zip(param_names, best_params):
    print(f"  {name:6s} = {val:.4f}")

# Estimate moment magnitude for best model
m0_best, mw_best = fwd.estimate_total_moment_and_mw(best_params)
print(f"\n✓ Best model moment:")
print(f"  M0 = {m0_best:.3e} N.m")
print(f"  Mw = {mw_best:.2f}")

# Compare with initial midpoint model
print(f"\n✓ Comparison: Initial vs Best")
print(f"  Initial misfit: {misfit_test:.6f}")
print(f"  Best misfit:    {result.best_model.misfit:.6f}")
improvement = (misfit_test - result.best_model.misfit) / misfit_test * 100
print(f"  Improvement:    {improvement:.1f}%")


STARTING NEIGHBOURHOOD ALGORITHM SEARCH
[NA] Adjusting n_samples_iteration from 30 to 35 to satisfy neighpy constraint (multiple of n_cells_resample=7).
[NA] Starting search: ni=100, ns=35, n=10, nr=7 -> expected evaluations=450
NAI - Initial Random Search
[NA] iter=000 eval=00001 misfit=2.769579e+02 best=2.769579e+02
[NA] iter=000 eval=00002 misfit=7.618012e+02 best=2.769579e+02
[NA] iter=000 eval=00003 misfit=2.285451e+02 best=2.285451e+02
[NA] iter=000 eval=00004 misfit=4.513069e+02 best=2.285451e+02
[NA] iter=000 eval=00005 misfit=6.275357e+02 best=2.285451e+02
[NA] iter=000 eval=00006 misfit=4.133923e+02 best=2.285451e+02
[NA] iter=000 eval=00007 misfit=4.572672e+02 best=2.285451e+02
[NA] iter=000 eval=00008 misfit=6.211029e+02 best=2.285451e+02
[NA] iter=000 eval=00009 misfit=6.818219e+02 best=2.285451e+02
[NA] iter=000 eval=00010 misfit=5.649620e+02 best=2.285451e+02
[NA] iter=000 eval=00011 misfit=3.688018e+02 best=2.285451e+02
[NA] iter=000 eval=00012 misfit=1.902730e+02 best=

## Step 5: Run Neighbourhood Algorithm (NA) Inversion

Configure and execute the NA search to find the best model parameters.


## Step 4: Compute Misfit and Build Misfit Calculator

Now calculate the misfit between observed and filtered synthetic waveforms.
